# **2. Motivos de Cartões e VAR: Embeddings, Grupos Semânticos e Resultados**

Responsabilidade deste notebook:

- carregar `data/processed/card_reasons.csv` e `data/processed/delay_reasons.csv`, gerados no notebook 1;
- consolidar as justificativas livres de cartões e explorar sua distribuição inicial;
- gerar embeddings, rodar BERTopic em duas passagens e fine-tunar o modelo a partir dos clusters da primeira passagem;
- rotular os clusters finais em grupos semânticos de motivos de cartões;
- produzir tabelas e gráficos de resultados gerais, por time e por árbitro;
- identificar menções textuais a VAR nos motivos de atraso/acréscimos e salvar os artefatos finais.


## **1. Bibliotecas**

Além das dependências de manipulação de dados e visualização, este notebook usa:

- `sentence-transformers` para gerar embeddings semânticos;
- `BERTopic`, `UMAP` e `HDBSCAN` para agrupar textos recorrentes;
- `scikit-learn` para medir similaridade entre grupos, clusters e consultas.


In [ ]:
import re
import shutil
import unicodedata
from itertools import combinations
from pathlib import Path

import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sentence_transformers import models as st_models
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 220)
plt.style.use("seaborn-v0_8-whitegrid")


## **2. Configuração**

Define caminhos, hiperparâmetros do modelo e limites de exibição para as tabelas comparativas. O modelo fine-tunado é reutilizado quando já existe em `models/`, evitando treinar novamente em execuções comuns.


In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
PROCESSED_PATH = DATA_PATH / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
VIZ_PATH = PROJECT_ROOT / "viz"
OUTPUT_DIR = PROCESSED_PATH / "semantic_reasons"
CARD_OUTPUT_DIR = OUTPUT_DIR / "card_reasons"
DELAY_OUTPUT_DIR = OUTPUT_DIR / "delay_reasons"

CARD_REASONS_PATH = PROCESSED_PATH / "card_reasons.csv"
DELAY_REASONS_PATH = PROCESSED_PATH / "delay_reasons.csv"

for directory in [MODELS_DIR, OUTPUT_DIR, CARD_OUTPUT_DIR, DELAY_OUTPUT_DIR, VIZ_PATH]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
FINETUNED_MODEL_DIR = MODELS_DIR / "football_card_reason_embeddings_two_pass"

RUN_FINE_TUNING = True
RETRAIN_FINETUNED_MODEL = False
USE_FINETUNED_IF_AVAILABLE = True
FINETUNE_EPOCHS = 1
FINETUNE_BATCH_SIZE = 32
MAX_TEXTS_PER_GROUP_FOR_FINETUNE = 220
MAX_TEXT_PAIRS_PER_GROUP_FOR_FINETUNE = 450

CARD_MIN_TOPIC_SIZE = 45
CARD_MIN_SAMPLES = 12
DELAY_MIN_TOPIC_SIZE = 35
DELAY_MIN_SAMPLES = 8
BERTOPIC_NGRAM_RANGE = (1, 3)

CARD_TEXT_COLUMN = "card_reason_text"
CARD_TEXT_KEY_COLUMN = "card_reason_text_key"
DELAY_TEXT_COLUMN = "delay_reason_clean"

TOP_N_CARD_TEXTS = 15
TOP_N_TEAMS = 15
TOP_N_REFEREES = 20


def save_plot(fig, filename: str) -> None:
    fig.savefig(VIZ_PATH / filename, dpi=300, bbox_inches="tight")


## **3. Carregamento dos Dados Limpos**

Os dois arquivos de entrada vêm do notebook 1. Para cartões, o texto usado no modelo é apenas `reason`, que corresponde à justificativa livre escrita pelo árbitro. Para atrasos/acréscimos, o texto usado é `delay_reason`.


In [ ]:
REQUIRED_CARD_COLUMNS = {
    "year", "round", "game_id", "game_number", "match_date", "home_team", "away_team", "arbitro",
    "card_type", "team", "minute", "half", "reason", "reason_key",
}
REQUIRED_DELAY_COLUMNS = {
    "year", "round", "game_id", "game_number", "match_date", "home_team", "away_team", "arbitro", "delay_reason",
}

cards = pd.read_csv(CARD_REASONS_PATH, keep_default_na=False)
delays = pd.read_csv(DELAY_REASONS_PATH, keep_default_na=False)

missing_cards = REQUIRED_CARD_COLUMNS.difference(cards.columns)
missing_delays = REQUIRED_DELAY_COLUMNS.difference(delays.columns)
if missing_cards:
    raise ValueError(f"Colunas ausentes em {CARD_REASONS_PATH.name}: {sorted(missing_cards)}")
if missing_delays:
    raise ValueError(f"Colunas ausentes em {DELAY_REASONS_PATH.name}: {sorted(missing_delays)}")

cards.head(3)


In [ ]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.casefold()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


cards[CARD_TEXT_COLUMN] = cards["reason"].fillna("").astype(str).str.strip()
cards[CARD_TEXT_KEY_COLUMN] = cards[CARD_TEXT_COLUMN].map(normalize_text)
cards = cards[cards[CARD_TEXT_KEY_COLUMN].ne("")].reset_index(drop=True)

delays[DELAY_TEXT_COLUMN] = delays["delay_reason"].fillna("").astype(str).str.strip()
delays[DELAY_TEXT_COLUMN] = delays[DELAY_TEXT_COLUMN].where(delays[DELAY_TEXT_COLUMN].ne(""), "n/a")
delays["delay_reason_key"] = delays[DELAY_TEXT_COLUMN].map(normalize_text)

pd.Series({
    "card_rows": len(cards),
    "unique_card_reason_texts": cards[CARD_TEXT_KEY_COLUMN].nunique(),
    "delay_rows": len(delays),
    "unique_delay_reason_texts": delays["delay_reason_key"].nunique(),
    "delay_na": delays[DELAY_TEXT_COLUMN].eq("n/a").sum(),
})


## **4. Leitura Inicial dos Cartões**

Antes de usar embeddings, vale medir o volume por ano, tipo de cartão e justificativas mais frequentes. Esta etapa ajuda a separar padrões muito repetidos de textos raros que tendem a virar outliers no HDBSCAN.


In [ ]:
card_dataset_summary = pd.Series({
    "cards": len(cards),
    "yellow_cards": int(cards["card_type"].eq("yellow").sum()),
    "red_cards": int(cards["card_type"].eq("red").sum()),
    "teams": cards["team"].nunique(),
    "referees": cards["arbitro"].nunique(),
    "unique_reason_texts": cards[CARD_TEXT_KEY_COLUMN].nunique(),
})
card_dataset_summary


In [ ]:
card_by_year_type = pd.crosstab(cards["year"], cards["card_type"])
card_by_year_type


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"yellow": "#E3B23C", "red": "#C44E52"}
card_by_year_type.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[colors.get(column, "#4C78A8") for column in card_by_year_type.columns],
)
ax.set_title("Cartões extraídos por ano e tipo")
ax.set_xlabel("Ano")
ax.set_ylabel("Cartões")
ax.legend(title="Tipo")
plt.tight_layout()
save_plot(fig, "cards_by_year_type.png")
plt.show()


In [ ]:
card_text_counts = (
    cards.groupby(CARD_TEXT_KEY_COLUMN)
    .agg(
        text_for_embedding=(CARD_TEXT_COLUMN, "first"),
        n_events=(CARD_TEXT_COLUMN, "size"),
        n_yellow=("card_type", lambda values: (values == "yellow").sum()),
        n_red=("card_type", lambda values: (values == "red").sum()),
    )
    .reset_index()
    .sort_values("n_events", ascending=False)
    .reset_index(drop=True)
)

card_texts = card_text_counts["text_for_embedding"].tolist()
card_text_counts.head(25)


In [ ]:
plot_top_texts = card_text_counts.head(TOP_N_CARD_TEXTS).sort_values("n_events")
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(plot_top_texts["text_for_embedding"], plot_top_texts["n_events"], color="#4C78A8")
ax.set_title("Justificativas livres de cartão mais recorrentes")
ax.set_xlabel("Cartões")
ax.set_ylabel("")
plt.tight_layout()
save_plot(fig, "top_card_reason_texts.png")
plt.show()


## **5. Funções de Tópicos e Embeddings**

Os helpers abaixo padronizam a criação do BERTopic, a sumarização dos clusters e a extração de exemplos por tópico. Eles são usados tanto na primeira quanto na segunda passagem do modelo.


In [ ]:
PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das", "de", "do", "dos", "e", "em", "na", "nas",
    "no", "nos", "o", "os", "ou", "para", "por", "que", "se", "sem", "um", "uma", "uns", "umas",
    "apos", "ate", "contra", "durante", "forma", "maneira", "motivo", "pela", "pelo", "pelas", "pelos",
    "num", "numa", "ter", "tendo", "dado", "haver", "havendo", "advertido", "advertida", "advertir",
    "adversario", "adversaria", "atleta", "jogador", "jogadora", "sr", "sra", "numero", "n", "equipe",
    "bola", "campo", "disputa", "jogada", "jogo", "partida", "seu", "sua", "seus", "suas", "foi", "foram",
    "acrescimo", "acrescimos", "devido", "inicio", "reinicio", "tempo", "minuto", "minutos", "jogadores",
}


def make_topic_model(min_topic_size: int, min_samples: int, n_neighbors: int = 20, n_components: int = 5) -> BERTopic:
    vectorizer_model = CountVectorizer(
        stop_words=list(PORTUGUESE_STOPWORDS),
        ngram_range=BERTOPIC_NGRAM_RANGE,
        min_df=2,
    )
    umap_model = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=min_topic_size,
        min_samples=min_samples,
        metric="euclidean",
        prediction_data=True,
    )
    return BERTopic(
        language="multilingual",
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False,
    )


def topic_words(topic_model: BERTopic, topic_id: int, top_n: int = 8) -> str:
    if topic_id == -1:
        return "outliers"
    words = topic_model.get_topic(topic_id) or []
    return ", ".join(word for word, _ in words[:top_n])


def build_topic_summary(topic_model: BERTopic, text_counts: pd.DataFrame, topic_col: str) -> pd.DataFrame:
    topic_info = topic_model.get_topic_info().rename(columns={"Topic": topic_col})
    event_counts = (
        text_counts.groupby(topic_col)
        .agg(
            n_unique_texts=("text_for_embedding", "size"),
            n_events=("n_events", "sum"),
        )
        .reset_index()
    )
    summary = topic_info.merge(event_counts, on=topic_col, how="left")
    summary["top_terms"] = summary[topic_col].map(lambda topic_id: topic_words(topic_model, int(topic_id)))
    return summary.sort_values("n_events", ascending=False).reset_index(drop=True)


def top_examples_by_topic(text_counts: pd.DataFrame, topic_col: str, top_n: int = 5) -> pd.DataFrame:
    columns = [topic_col, "n_events", "text_for_embedding"]
    return (
        text_counts.sort_values([topic_col, "n_events"], ascending=[True, False])
        .groupby(topic_col)
        .head(top_n)
        [columns]
        .reset_index(drop=True)
    )


## **6. Passagem 1: Modelo Base + BERTopic**

A primeira passagem usa o modelo base para gerar embeddings e deixar o BERTopic encontrar clusters sem rótulos manuais. Esses clusters são usados apenas como material de treino para adaptar o espaço vetorial ao vocabulário das súmulas.


In [ ]:
def load_embedding_model(model_name_or_path) -> SentenceTransformer:
    path = str(model_name_or_path)
    try:
        model = SentenceTransformer(path)
        print(f"[OK] SentenceTransformer: {path}")
        return model
    except Exception:
        word_model = st_models.Transformer(path, max_seq_length=256)
        pooling = st_models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True,
        )
        model = SentenceTransformer(modules=[word_model, pooling])
        print(f"[OK] Transformer + mean pooling: {path}")
        return model


def model_dir_ready(path: Path) -> bool:
    return (path / "modules.json").exists()


base_model = load_embedding_model(MODEL_NAME)

base_card_embeddings = base_model.encode(
    card_texts,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
base_card_embeddings = np.asarray(base_card_embeddings)
print(f"Embeddings base: {base_card_embeddings.shape}")


In [ ]:
card_topic_model_pass1 = make_topic_model(
    min_topic_size=CARD_MIN_TOPIC_SIZE,
    min_samples=CARD_MIN_SAMPLES,
    n_neighbors=25,
    n_components=5,
)
card_topics_pass1, _ = card_topic_model_pass1.fit_transform(card_texts, embeddings=base_card_embeddings)
card_text_counts["bertopic_topic_pass1"] = card_topics_pass1

pass1_counts = card_text_counts.rename(columns={"bertopic_topic_pass1": "bertopic_topic"})
pass1_summary = build_topic_summary(card_topic_model_pass1, pass1_counts, "bertopic_topic")
pass1_summary = pass1_summary.rename(columns={"bertopic_topic": "bertopic_topic_pass1"})

print(f"Pass 1 — clusters encontrados: {pass1_summary['bertopic_topic_pass1'].ne(-1).sum()}")
pass1_summary.head(20)


## **7. Fine-Tuning a Partir dos Clusters da Passagem 1**

Textos dentro do mesmo cluster da primeira passagem são tratados como pares positivos. O objetivo é aproximar expressões equivalentes do domínio futebolístico sem criar categorias finais por regra manual.


In [ ]:
def build_pairs_from_clusters(
    text_counts: pd.DataFrame,
    topic_col: str,
    max_texts: int = MAX_TEXTS_PER_GROUP_FOR_FINETUNE,
    max_pairs: int = MAX_TEXT_PAIRS_PER_GROUP_FOR_FINETUNE,
) -> list[dict]:
    pairs: list[dict] = []
    for _, group in text_counts[text_counts[topic_col].ne(-1)].groupby(topic_col):
        texts = (
            group.sort_values("n_events", ascending=False)["text_for_embedding"]
            .dropna()
            .astype(str)
            .str.strip()
            .loc[lambda values: values.ne("")]
            .drop_duplicates()
            .head(max_texts)
            .tolist()
        )
        pairs.extend(
            {"anchor": left, "positive": right}
            for left, right in list(combinations(texts, 2))[:max_pairs]
        )
    return pairs


if RUN_FINE_TUNING:
    if USE_FINETUNED_IF_AVAILABLE and model_dir_ready(FINETUNED_MODEL_DIR) and not RETRAIN_FINETUNED_MODEL:
        print(f"Modelo fine-tunado encontrado em {FINETUNED_MODEL_DIR}. Carregando.")
        embedding_model = load_embedding_model(FINETUNED_MODEL_DIR)
    else:
        import datasets as hf_datasets
        from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
        from sentence_transformers.losses import MultipleNegativesRankingLoss
        from sentence_transformers.training_args import BatchSamplers
        import sentence_transformers.base.trainer as st_base_trainer

        if not hasattr(st_base_trainer, "IterableDataset"):
            from torch.utils.data import IterableDataset as TorchIterableDataset
            st_base_trainer.IterableDataset = TorchIterableDataset

        if FINETUNED_MODEL_DIR.exists() and RETRAIN_FINETUNED_MODEL:
            shutil.rmtree(FINETUNED_MODEL_DIR)

        positive_pairs = build_pairs_from_clusters(card_text_counts, "bertopic_topic_pass1")
        if not positive_pairs:
            raise ValueError("Nenhum par positivo foi gerado a partir dos clusters da Passagem 1.")

        print(f"Pares positivos gerados dos clusters da Passagem 1: {len(positive_pairs):,}")
        train_dataset = hf_datasets.Dataset.from_list(positive_pairs)

        ft_model = load_embedding_model(MODEL_NAME)
        train_loss = MultipleNegativesRankingLoss(ft_model)
        training_args = SentenceTransformerTrainingArguments(
            output_dir=str(FINETUNED_MODEL_DIR / "checkpoints"),
            num_train_epochs=FINETUNE_EPOCHS,
            per_device_train_batch_size=FINETUNE_BATCH_SIZE,
            learning_rate=2e-5,
            warmup_ratio=0.1,
            fp16=False,
            use_cpu=True,
            batch_sampler=BatchSamplers.NO_DUPLICATES,
            save_strategy="no",
            logging_steps=50,
        )
        trainer = SentenceTransformerTrainer(
            model=ft_model,
            args=training_args,
            train_dataset=train_dataset,
            loss=train_loss,
        )
        trainer.train()
        ft_model.save(str(FINETUNED_MODEL_DIR))
        embedding_model = ft_model
        print(f"Modelo salvo em: {FINETUNED_MODEL_DIR}")
else:
    embedding_model = base_model
    print("Fine-tuning desabilitado. Usando modelo base.")

print(f"Dimensão dos embeddings: {embedding_model.get_sentence_embedding_dimension()}")


## **8. Passagem 2: Clusters Finais de Cartões**

A segunda passagem usa o modelo fine-tunado para recalcular embeddings e clusters. Estes são os clusters que serão inspecionados, rotulados e propagados para cada cartão individual.


In [ ]:
card_embeddings = embedding_model.encode(
    card_texts,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
card_embeddings = np.asarray(card_embeddings)
print(f"Embeddings de cartões: {card_embeddings.shape}")


In [ ]:
card_topic_model = make_topic_model(
    min_topic_size=CARD_MIN_TOPIC_SIZE,
    min_samples=CARD_MIN_SAMPLES,
    n_neighbors=25,
    n_components=5,
)
card_topics, _ = card_topic_model.fit_transform(card_texts, embeddings=card_embeddings)
card_text_counts["bertopic_topic"] = card_topics

card_topic_summary = build_topic_summary(card_topic_model, card_text_counts, "bertopic_topic")
print(f"Pass 2 — clusters encontrados: {card_topic_summary['bertopic_topic'].ne(-1).sum()}")
card_topic_summary.head(20)


In [ ]:
card_topic_examples = top_examples_by_topic(card_text_counts, "bertopic_topic", top_n=5)
card_topic_examples.head(30)


## **9. Rotulagem Semântica dos Motivos de Cartão**

A rotulagem combina inspeção dos clusters com frases-semente por grupo. Cada cluster recebe o grupo cujo centroide está semanticamente mais próximo; clusters abaixo do limiar ficam em `Outros`. Os outliers do HDBSCAN passam por uma segunda tentativa usando similaridade direta entre texto e grupo.


In [ ]:
def print_cluster_inspection(
    topic_model: BERTopic,
    text_counts: pd.DataFrame,
    topic_col: str,
    top_n_examples: int = 3,
) -> None:
    summary = build_topic_summary(topic_model, text_counts, topic_col)
    examples_df = top_examples_by_topic(text_counts, topic_col, top_n=top_n_examples)

    print(f"{'ID':>4}  {'Eventos':>8}  Top terms")
    print("-" * 110)
    for _, row in summary.iterrows():
        topic_id = int(row[topic_col])
        n_events = int(row.get("n_events", 0))
        terms = str(row["top_terms"])
        suffix = "  [outliers HDBSCAN]" if topic_id == -1 else ""
        print(f"\n{topic_id:>4}  {n_events:>8,}  {terms}{suffix}")
        examples = examples_df[examples_df[topic_col].eq(topic_id)]["text_for_embedding"].tolist()
        for example in examples:
            print(f"              > {str(example)[:108]}")


print_cluster_inspection(card_topic_model, card_text_counts, "bertopic_topic")


In [ ]:
GROUP_PROTOTYPES: dict[str, list[str]] = {
    "Ação temerária": [
        "calçar adversário de maneira temerária na disputa de bola",
        "dar entrada temerária no adversário",
        "golpear adversário de maneira temerária na disputa de bola",
        "rasteira temerária contra adversário na disputa de bola",
        "entrada contra adversário de forma temerária",
        "dar um calço temerário na disputa de bola",
        "golpear adversário com o braço de forma temerária na disputa de bola",
        "atingir adversário com o braço de maneira temerária na disputa de bola",
    ],
    "Ataque promissor / falta tática": [
        "cometer falta tática impedindo ataque promissor",
        "impedir oportunidade clara de gol",
        "segurar adversário impedindo ataque promissor",
        "falta tática para impedir ataque promissor",
        "mão na bola impedindo ataque promissor",
        "segurar adversário na disputa de bola impedindo ataque promissor",
        "agarrar adversário interrompendo ataque promissor",
    ],
    "Protesto contra arbitragem": [
        "reclamar das decisões da arbitragem",
        "desaprovar com palavras e gestos as decisões da arbitragem",
        "protestar de forma acintosa contra as decisões da arbitragem",
        "reclamar contra as marcações da arbitragem",
        "reclamar das decisões da arbitragem de maneira acintosa do banco de reservas",
        "se levantar do banco de reservas e protestar de maneira acintosa",
        "conduta antidesportiva ao protestar de forma acintosa contra as decisões da arbitragem",
        "conduta antidesportiva ao reclamar de forma acintosa contra as decisões da arbitragem",
    ],
    "Conduta antidesportiva/regras": [
        "conduta antidesportiva",
        "empurrar adversário com bola fora de jogo",
        "infringir persistentemente as regras do jogo",
        "simular falta para enganar árbitro",
        "segurar adversário de forma acintosa desrespeitando o jogo",
    ],
    "Retardar/reiniciar jogo": [
        "retardar o reinício de jogo",
        "chutar bola para longe com jogo paralisado",
        "retardar arremesso lateral ou tiro livre",
        "retardar saída de campo durante substituição",
        "demorar para reiniciar o jogo",
    ],
    "Comemoração/entrada em campo": [
        "tirar camisa na comemoração de gol",
        "entrar em campo para comemorar gol sem permissão do árbitro",
    ],
    "Força excessiva / jogo brusco grave": [
        "golpear rosto do adversário de maneira temerária",
        "força excessiva contra adversário fora da disputa de bola",
        "golpear adversário com o braço no rosto",
    ],
}

SIMILARITY_THRESHOLD = 0.55
OUTLIER_THRESHOLD = 0.45


In [ ]:
def compute_group_centroids(
    group_prototypes: dict[str, list[str]],
    model: SentenceTransformer,
) -> dict[str, np.ndarray]:
    centroids = {}
    for group_name, phrases in group_prototypes.items():
        embeddings = np.asarray(model.encode(phrases, normalize_embeddings=True, show_progress_bar=False))
        centroid = embeddings.mean(axis=0)
        centroids[group_name] = centroid / np.linalg.norm(centroid)
    return centroids


def compute_cluster_centroids(
    text_counts: pd.DataFrame,
    embeddings: np.ndarray,
    all_texts: list[str],
    topic_col: str,
) -> dict[int, np.ndarray]:
    text_to_idx = {text: index for index, text in enumerate(all_texts)}
    cluster_centroids = {}
    for topic_id, group_df in text_counts.groupby(topic_col):
        indices = [text_to_idx[text] for text in group_df["text_for_embedding"] if text in text_to_idx]
        if not indices:
            continue
        centroid = embeddings[indices].mean(axis=0)
        norm = np.linalg.norm(centroid)
        cluster_centroids[int(topic_id)] = centroid / norm if norm > 0 else centroid
    return cluster_centroids


def auto_assign_clusters(
    cluster_centroids: dict[int, np.ndarray],
    group_centroids: dict[str, np.ndarray],
    threshold: float,
) -> tuple[dict[int, str], pd.DataFrame]:
    group_names = list(group_centroids.keys())
    group_matrix = np.stack([group_centroids[group] for group in group_names])
    topic_to_group = {-1: "Outros"}
    rows = []
    for topic_id, cluster_centroid in sorted(cluster_centroids.items()):
        if topic_id == -1:
            continue
        similarities = cosine_similarity(cluster_centroid.reshape(1, -1), group_matrix).ravel()
        best_idx = int(np.argmax(similarities))
        best_similarity = float(similarities[best_idx])
        assigned_group = group_names[best_idx] if best_similarity >= threshold else "Outros"
        topic_to_group[topic_id] = assigned_group
        rows.append({
            "topic_id": topic_id,
            "assigned_group": assigned_group,
            "best_similarity": round(best_similarity, 4),
            "best_group": group_names[best_idx],
        })
    return topic_to_group, pd.DataFrame(rows)


def assign_outliers_by_similarity(
    text_counts: pd.DataFrame,
    embeddings: np.ndarray,
    all_texts: list[str],
    topic_col: str,
    group_centroids: dict[str, np.ndarray],
    threshold: float,
) -> pd.Series:
    group_names = list(group_centroids.keys())
    group_matrix = np.stack([group_centroids[group] for group in group_names])
    text_to_idx = {text: index for index, text in enumerate(all_texts)}
    outlier_mask = text_counts[topic_col].eq(-1)
    results = pd.Series("Outros", index=text_counts.index, dtype=str)

    for row_idx, row in text_counts[outlier_mask].iterrows():
        embedding_idx = text_to_idx.get(row["text_for_embedding"])
        if embedding_idx is None:
            continue
        similarities = cosine_similarity(embeddings[embedding_idx].reshape(1, -1), group_matrix).ravel()
        best_idx = int(np.argmax(similarities))
        best_similarity = float(similarities[best_idx])
        results.at[row_idx] = group_names[best_idx] if best_similarity >= threshold else "Outros"

    return results


group_centroids = compute_group_centroids(GROUP_PROTOTYPES, embedding_model)
cluster_centroids_map = compute_cluster_centroids(card_text_counts, card_embeddings, card_texts, "bertopic_topic")
TOPIC_TO_GROUP, assignment_report = auto_assign_clusters(cluster_centroids_map, group_centroids, SIMILARITY_THRESHOLD)

card_text_counts["card_reason_group"] = card_text_counts["bertopic_topic"].map(TOPIC_TO_GROUP).fillna("Outros")
outlier_mask = card_text_counts["bertopic_topic"].eq(-1)
outlier_groups = assign_outliers_by_similarity(
    card_text_counts,
    card_embeddings,
    card_texts,
    "bertopic_topic",
    group_centroids,
    OUTLIER_THRESHOLD,
)
card_text_counts.loc[outlier_mask, "card_reason_group"] = outlier_groups[outlier_mask]

group_map = card_text_counts.set_index(CARD_TEXT_KEY_COLUMN)["card_reason_group"]
cards["card_reason_group"] = cards[CARD_TEXT_KEY_COLUMN].map(group_map).fillna("Outros")
CARD_GROUPS = sorted(cards["card_reason_group"].unique())

outros_clusters = int(assignment_report["assigned_group"].eq("Outros").sum())
outliers_total = int(outlier_mask.sum())
outliers_assigned = int(card_text_counts.loc[outlier_mask, "card_reason_group"].ne("Outros").sum())
print(f"Clusters mapeados automaticamente: {len(assignment_report) - outros_clusters} / {len(assignment_report)}")
print(f"Outliers HDBSCAN reclassificados: {outliers_assigned} / {outliers_total} textos únicos")
print(f"Eventos restantes em Outros: {cards['card_reason_group'].eq('Outros').sum()}")
assignment_report.sort_values("best_similarity", ascending=False)


In [ ]:
def diagnose_outros(
    assignment_report: pd.DataFrame,
    topic_model: BERTopic,
    text_counts: pd.DataFrame,
    topic_col: str,
    top_n_examples: int = 5,
    top_n_outliers: int = 20,
) -> None:
    examples_df = top_examples_by_topic(text_counts, topic_col, top_n=top_n_examples)
    missed = assignment_report[assignment_report["assigned_group"].eq("Outros")].sort_values("best_similarity", ascending=False)

    print("=" * 100)
    print(f"Clusters abaixo do threshold: {len(missed)}")
    print("=" * 100)
    for _, row in missed.iterrows():
        topic_id = int(row["topic_id"])
        print(f"\ntopic_id={topic_id} | best_similarity={row['best_similarity']:.4f} | best_group='{row['best_group']}'")
        print(f"top terms: {topic_words(topic_model, topic_id)}")
        for example in examples_df[examples_df[topic_col].eq(topic_id)]["text_for_embedding"].tolist():
            print(f"  > {str(example)[:110]}")

    print("\n" + "=" * 100)
    print(f"Outliers HDBSCAN mais frequentes: top {top_n_outliers}")
    print("=" * 100)
    outlier_rows = (
        text_counts[text_counts[topic_col].eq(-1)]
        .sort_values("n_events", ascending=False)
        .head(top_n_outliers)
        [["n_events", "text_for_embedding"]]
    )
    for _, row in outlier_rows.iterrows():
        print(f"n={int(row['n_events']):>4} | {str(row['text_for_embedding'])[:110]}")


diagnose_outros(assignment_report, card_topic_model, card_text_counts, "bertopic_topic")


## **10. Resultados Gerais dos Motivos de Cartão**

A partir daqui, cada cartão já tem um grupo semântico final em `card_reason_group`. As primeiras tabelas mostram o peso de cada grupo no total, por ano e por tipo de cartão.


In [ ]:
card_group_summary = (
    cards.groupby("card_reason_group")
    .agg(
        n_events=("card_reason_group", "size"),
        n_yellow=("card_type", lambda values: values.eq("yellow").sum()),
        n_red=("card_type", lambda values: values.eq("red").sum()),
        n_unique_texts=(CARD_TEXT_KEY_COLUMN, "nunique"),
    )
    .reset_index()
    .sort_values("n_events", ascending=False)
)
card_group_summary["pct_events"] = (card_group_summary["n_events"] / len(cards)).round(4)
card_group_summary


In [ ]:
card_group_by_year = pd.crosstab(cards["year"], cards["card_reason_group"], normalize="index").round(3)
card_group_by_year


In [ ]:
card_group_by_type = pd.crosstab(cards["card_type"], cards["card_reason_group"])
card_group_by_type


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
plot_card_summary = card_group_summary.sort_values("n_events", ascending=True)
axes[0].barh(plot_card_summary["card_reason_group"], plot_card_summary["n_events"], color="#4C78A8")
axes[0].set_title("Cartões por grupo semântico")
axes[0].set_xlabel("Cartões")
axes[0].set_ylabel("")

top_groups = card_group_summary.head(5)["card_reason_group"].tolist()
card_group_by_year[top_groups].plot(ax=axes[1], marker="o")
axes[1].set_title("Principais grupos por ano")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Proporção dos cartões")
axes[1].legend(title="Grupo", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
save_plot(fig, "card_reason_groups_overview.png")
plt.show()


## **11. Motivos de Cartão por Time**

As tabelas abaixo contam os grupos de motivos para cada time que recebeu cartão. A tabela compacta mostra o grupo mais frequente por time; o gráfico mostra a composição proporcional dos grupos entre os times com mais cartões.


In [ ]:
card_reason_by_team = (
    cards.groupby(["team", "card_reason_group"])
    .agg(
        n_events=("card_reason_group", "size"),
        n_yellow=("card_type", lambda values: values.eq("yellow").sum()),
        n_red=("card_type", lambda values: values.eq("red").sum()),
        n_unique_texts=(CARD_TEXT_KEY_COLUMN, "nunique"),
    )
    .reset_index()
)
team_card_totals = cards.groupby("team").size().rename("total_cards")
card_reason_by_team = card_reason_by_team.merge(team_card_totals, on="team", how="left")
card_reason_by_team["pct_team_cards"] = (card_reason_by_team["n_events"] / card_reason_by_team["total_cards"]).round(4)
card_reason_by_team = card_reason_by_team.sort_values(["total_cards", "team", "n_events"], ascending=[False, True, False])

card_reason_top_by_team = (
    card_reason_by_team.sort_values(["team", "n_events", "card_reason_group"], ascending=[True, False, True])
    .groupby("team")
    .head(1)
    .sort_values("total_cards", ascending=False)
    .reset_index(drop=True)
)
card_reason_top_by_team.head(30)


In [ ]:
top_card_teams = cards["team"].value_counts().head(TOP_N_TEAMS).index
team_group_plot = pd.crosstab(
    cards.loc[cards["team"].isin(top_card_teams), "team"],
    cards.loc[cards["team"].isin(top_card_teams), "card_reason_group"],
    normalize="index",
).loc[top_card_teams]

fig, ax = plt.subplots(figsize=(12, 8))
team_group_plot.plot(kind="barh", stacked=True, ax=ax, colormap="tab20")
ax.invert_yaxis()
ax.set_title(f"Composição dos motivos de cartão nos {TOP_N_TEAMS} times com mais cartões")
ax.set_xlabel("Proporção dos cartões do time")
ax.set_ylabel("")
ax.legend(title="Grupo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_plot(fig, "card_reason_groups_by_team.png")
plt.show()


## **12. Motivos de Cartão por Árbitro**

Aqui a unidade é o árbitro da partida. A tabela compacta mostra qual grupo de motivo aparece com mais frequência nos cartões registrados em jogos apitados por cada árbitro.


In [ ]:
card_reason_by_referee = (
    cards.groupby(["arbitro", "card_reason_group"])
    .agg(
        n_events=("card_reason_group", "size"),
        n_yellow=("card_type", lambda values: values.eq("yellow").sum()),
        n_red=("card_type", lambda values: values.eq("red").sum()),
        n_unique_texts=(CARD_TEXT_KEY_COLUMN, "nunique"),
    )
    .reset_index()
)
referee_card_totals = cards.groupby("arbitro").size().rename("total_cards")
card_reason_by_referee = card_reason_by_referee.merge(referee_card_totals, on="arbitro", how="left")
card_reason_by_referee["pct_referee_cards"] = (card_reason_by_referee["n_events"] / card_reason_by_referee["total_cards"]).round(4)
card_reason_by_referee = card_reason_by_referee.sort_values(["total_cards", "arbitro", "n_events"], ascending=[False, True, False])

card_reason_top_by_referee = (
    card_reason_by_referee.sort_values(["arbitro", "n_events", "card_reason_group"], ascending=[True, False, True])
    .groupby("arbitro")
    .head(1)
    .sort_values("total_cards", ascending=False)
    .reset_index(drop=True)
)
card_reason_top_by_referee.head(30)


In [ ]:
top_card_referees = cards["arbitro"].value_counts().head(TOP_N_REFEREES).index
referee_group_plot = pd.crosstab(
    cards.loc[cards["arbitro"].isin(top_card_referees), "arbitro"],
    cards.loc[cards["arbitro"].isin(top_card_referees), "card_reason_group"],
    normalize="index",
).loc[top_card_referees]

fig, ax = plt.subplots(figsize=(12, 9))
referee_group_plot.plot(kind="barh", stacked=True, ax=ax, colormap="tab20")
ax.invert_yaxis()
ax.set_title(f"Composição dos motivos de cartão nos {TOP_N_REFEREES} árbitros com mais cartões")
ax.set_xlabel("Proporção dos cartões em jogos do árbitro")
ax.set_ylabel("")
ax.legend(title="Grupo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_plot(fig, "card_reason_groups_by_referee.png")
plt.show()


## **13. Busca Semântica nos Motivos de Cartão**

Esta célula permite testar consultas livres contra os textos únicos de justificativas. Ela é útil para auditar se expressões semelhantes ficaram próximas no espaço de embeddings.


In [ ]:
def embed_query(text: str) -> np.ndarray:
    return np.asarray(embedding_model.encode([text], normalize_embeddings=True))


def semantic_search_cards(query: str, top_k: int = 10) -> pd.DataFrame:
    query_embedding = embed_query(query)
    similarities = cosine_similarity(query_embedding, card_embeddings).ravel()
    result = card_text_counts.copy()
    result["similarity"] = similarities
    return result.sort_values("similarity", ascending=False)[[
        "similarity", "n_events", "card_reason_group", "bertopic_topic", "text_for_embedding",
    ]].head(top_k)


semantic_search_cards("reclamação contra arbitragem", top_k=10)


## **14. Motivos de Atraso/Acréscimos e VAR**

Os motivos de atraso/acréscimos são tratados separadamente dos cartões. O BERTopic ajuda a inspecionar padrões textuais, enquanto a marcação de VAR é feita por sinais lexicais explícitos como `VAR`, `AVAR`, `checagem`, `revisão` e `monitor`.


In [ ]:
delay_text_counts = (
    delays[delays[DELAY_TEXT_COLUMN].ne("n/a")]
    .groupby("delay_reason_key")
    .agg(
        text_for_embedding=(DELAY_TEXT_COLUMN, "first"),
        n_games=(DELAY_TEXT_COLUMN, "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .reset_index()
    .sort_values("n_games", ascending=False)
    .reset_index(drop=True)
)
delay_text_counts.head(25)


In [ ]:
delay_texts = delay_text_counts["text_for_embedding"].tolist()
delay_embeddings = embedding_model.encode(
    delay_texts,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=64,
)
delay_embeddings = np.asarray(delay_embeddings)
print(f"Embeddings de atrasos/acréscimos: {delay_embeddings.shape}")


In [ ]:
delay_topic_model = make_topic_model(
    min_topic_size=DELAY_MIN_TOPIC_SIZE,
    min_samples=DELAY_MIN_SAMPLES,
    n_neighbors=25,
    n_components=5,
)
delay_topics, _ = delay_topic_model.fit_transform(delay_texts, embeddings=delay_embeddings)
delay_text_counts["bertopic_topic"] = delay_topics

delay_text_counts_for_summary = delay_text_counts.rename(columns={"n_games": "n_events"})
delay_topic_summary = build_topic_summary(delay_topic_model, delay_text_counts_for_summary, "bertopic_topic")
delay_topic_summary.head(20)


In [ ]:
delay_topic_examples = top_examples_by_topic(
    delay_text_counts.rename(columns={"n_games": "n_events"}),
    "bertopic_topic",
    top_n=5,
).rename(columns={"n_events": "n_games"})
delay_topic_examples.head(30)


In [ ]:
VAR_SIGNAL_PATTERNS = {
    "var": r"\bvar\b",
    "avar": r"\bavar\b",
    "vor": r"\bvor\b|sala\s+vor",
    "ara": r"\bara\b",
    "arbitro_de_video": r"arbitro\s+de\s+video|arbitragem\s+de\s+video|video\s+arbitro",
    "checagem": r"checag(?:em|ens)|checando|checar",
    "revisao": r"revis(?:ao|oes|ar|ado|ada)|review",
    "monitor_cabine": r"monitor|cabine",
}


def detect_var_signals(text: str) -> tuple[bool, str]:
    normalized = normalize_text(text)
    matches = [name for name, pattern in VAR_SIGNAL_PATTERNS.items() if re.search(pattern, normalized)]
    return bool(matches), ", ".join(matches)


var_detection = delays[DELAY_TEXT_COLUMN].apply(detect_var_signals)
delays["var_used"] = var_detection.map(lambda item: item[0])
delays["var_signal"] = var_detection.map(lambda item: item[1])

var_summary = pd.Series({
    "games_total": len(delays),
    "games_with_var_signal": int(delays["var_used"].sum()),
    "pct_with_var_signal": round(delays["var_used"].mean(), 4),
})
var_summary


In [ ]:
delay_var_by_year = (
    delays.groupby("year")
    .agg(
        games=("game_id", "count"),
        games_with_var_signal=("var_used", "sum"),
    )
    .reset_index()
)
delay_var_by_year["pct_with_var_signal"] = (
    delay_var_by_year["games_with_var_signal"] / delay_var_by_year["games"]
).round(4)
delay_var_by_year


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(delay_var_by_year["year"], delay_var_by_year["pct_with_var_signal"], marker="o", color="#4C78A8")
ax.set_title("Partidas com menção de VAR em delay_reason")
ax.set_xlabel("Ano")
ax.set_ylabel("Proporção")
ax.set_ylim(0, max(0.05, delay_var_by_year["pct_with_var_signal"].max() * 1.15))
plt.tight_layout()
save_plot(fig, "var_mentions_by_year.png")
plt.show()


In [ ]:
var_examples = (
    delays[delays["var_used"]]
    .sort_values(["year", "round", "game_number"])
    [["year", "round", "home_team", "away_team", "arbitro", "var_signal", DELAY_TEXT_COLUMN]]
    .head(25)
)
var_examples


## **15. Uso de VAR por Time e por Árbitro**

Para times, a contagem considera partidas em que o time aparece como mandante ou visitante. Para árbitros, a contagem considera as partidas registradas em `delay_reasons`. Em ambos os casos, `var_used` significa presença de sinal textual de VAR no campo de atraso/acréscimos, não uma validação externa do protocolo da partida.


In [ ]:
def most_common_non_empty(values: pd.Series) -> str:
    cleaned = values.astype(str).str.strip()
    cleaned = cleaned[cleaned.ne("")]
    if cleaned.empty:
        return ""
    return cleaned.value_counts().idxmax()


delay_home_games = delays.rename(columns={"home_team": "team", "away_team": "opponent"})[
    ["year", "round", "game_id", "game_number", "team", "opponent", "arbitro", "var_used", "var_signal", DELAY_TEXT_COLUMN]
]
delay_away_games = delays.rename(columns={"away_team": "team", "home_team": "opponent"})[
    ["year", "round", "game_id", "game_number", "team", "opponent", "arbitro", "var_used", "var_signal", DELAY_TEXT_COLUMN]
]
delay_team_games = pd.concat([delay_home_games, delay_away_games], ignore_index=True)

delay_var_by_team = (
    delay_team_games.groupby("team")
    .agg(
        games=("game_id", "count"),
        games_with_var_signal=("var_used", "sum"),
        top_var_signal=("var_signal", most_common_non_empty),
    )
    .reset_index()
)
delay_var_by_team["pct_with_var_signal"] = (
    delay_var_by_team["games_with_var_signal"] / delay_var_by_team["games"]
).round(4)
delay_var_by_team = delay_var_by_team.sort_values(["games", "games_with_var_signal"], ascending=False).reset_index(drop=True)
delay_var_by_team.head(30)


In [ ]:
top_var_teams = delay_var_by_team.head(TOP_N_TEAMS).sort_values("pct_with_var_signal")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_var_teams["team"], top_var_teams["pct_with_var_signal"], color="#59A14F")
ax.set_title(f"Menções de VAR nos {TOP_N_TEAMS} times com mais partidas")
ax.set_xlabel("Proporção de partidas com sinal de VAR")
ax.set_ylabel("")
plt.tight_layout()
save_plot(fig, "var_mentions_by_team.png")
plt.show()


In [ ]:
delay_var_by_referee = (
    delays.groupby("arbitro")
    .agg(
        games=("game_id", "count"),
        games_with_var_signal=("var_used", "sum"),
        top_var_signal=("var_signal", most_common_non_empty),
    )
    .reset_index()
)
delay_var_by_referee["pct_with_var_signal"] = (
    delay_var_by_referee["games_with_var_signal"] / delay_var_by_referee["games"]
).round(4)
delay_var_by_referee = delay_var_by_referee.sort_values(["games", "games_with_var_signal"], ascending=False).reset_index(drop=True)
delay_var_by_referee.head(30)


In [ ]:
top_var_referees = delay_var_by_referee.head(TOP_N_REFEREES).sort_values("pct_with_var_signal")
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_var_referees["arbitro"], top_var_referees["pct_with_var_signal"], color="#59A14F")
ax.set_title(f"Menções de VAR nos {TOP_N_REFEREES} árbitros com mais partidas")
ax.set_xlabel("Proporção de partidas com sinal de VAR")
ax.set_ylabel("")
plt.tight_layout()
save_plot(fig, "var_mentions_by_referee.png")
plt.show()


## **16. Salvamento dos Artefatos**

Os arquivos salvos mantêm os artefatos de modelagem e acrescentam tabelas analíticas por time e por árbitro. Assim, a leitura final não depende apenas dos gráficos do notebook.


In [ ]:
outros_audit = (
    cards[cards["card_reason_group"].eq("Outros")]
    .groupby(CARD_TEXT_COLUMN)
    .size()
    .reset_index(name="n_events")
    .sort_values("n_events", ascending=False)
)

card_topic_summary_path = CARD_OUTPUT_DIR / "card_bertopic_topic_summary.csv"
card_topic_examples_path = CARD_OUTPUT_DIR / "card_bertopic_topic_examples.csv"
card_unique_topics_path = CARD_OUTPUT_DIR / "card_reason_unique_topics_labels.csv"
card_labeled_path = CARD_OUTPUT_DIR / "card_reasons_with_labels.csv"
card_group_summary_path = CARD_OUTPUT_DIR / "card_reason_group_summary.csv"
card_group_by_year_path = CARD_OUTPUT_DIR / "card_reason_group_by_year.csv"
card_group_by_type_path = CARD_OUTPUT_DIR / "card_reason_group_by_card_type.csv"
card_reason_by_team_path = CARD_OUTPUT_DIR / "card_reason_by_team.csv"
card_reason_top_by_team_path = CARD_OUTPUT_DIR / "card_reason_top_by_team.csv"
card_reason_by_referee_path = CARD_OUTPUT_DIR / "card_reason_by_referee.csv"
card_reason_top_by_referee_path = CARD_OUTPUT_DIR / "card_reason_top_by_referee.csv"
outros_audit_path = CARD_OUTPUT_DIR / "card_reason_outros_audit.csv"

delay_topic_summary_path = DELAY_OUTPUT_DIR / "delay_bertopic_topic_summary.csv"
delay_topic_examples_path = DELAY_OUTPUT_DIR / "delay_bertopic_topic_examples.csv"
delay_unique_topics_path = DELAY_OUTPUT_DIR / "delay_reason_unique_topics.csv"
delay_labeled_path = DELAY_OUTPUT_DIR / "delay_reasons_with_var.csv"
delay_var_by_year_path = DELAY_OUTPUT_DIR / "delay_var_by_year.csv"
delay_var_by_team_path = DELAY_OUTPUT_DIR / "delay_var_by_team.csv"
delay_var_by_referee_path = DELAY_OUTPUT_DIR / "delay_var_by_referee.csv"

card_topic_summary.to_csv(card_topic_summary_path, index=False)
card_topic_examples.to_csv(card_topic_examples_path, index=False)
card_text_counts.to_csv(card_unique_topics_path, index=False)
cards.to_csv(card_labeled_path, index=False)
card_group_summary.to_csv(card_group_summary_path, index=False)
card_group_by_year.to_csv(card_group_by_year_path)
card_group_by_type.to_csv(card_group_by_type_path)
card_reason_by_team.to_csv(card_reason_by_team_path, index=False)
card_reason_top_by_team.to_csv(card_reason_top_by_team_path, index=False)
card_reason_by_referee.to_csv(card_reason_by_referee_path, index=False)
card_reason_top_by_referee.to_csv(card_reason_top_by_referee_path, index=False)
outros_audit.to_csv(outros_audit_path, index=False)

delay_topic_summary.to_csv(delay_topic_summary_path, index=False)
delay_topic_examples.to_csv(delay_topic_examples_path, index=False)
delay_text_counts.to_csv(delay_unique_topics_path, index=False)
delays.to_csv(delay_labeled_path, index=False)
delay_var_by_year.to_csv(delay_var_by_year_path, index=False)
delay_var_by_team.to_csv(delay_var_by_team_path, index=False)
delay_var_by_referee.to_csv(delay_var_by_referee_path, index=False)

output_paths = [
    card_topic_summary_path,
    card_topic_examples_path,
    card_unique_topics_path,
    card_labeled_path,
    card_group_summary_path,
    card_group_by_year_path,
    card_group_by_type_path,
    card_reason_by_team_path,
    card_reason_top_by_team_path,
    card_reason_by_referee_path,
    card_reason_top_by_referee_path,
    outros_audit_path,
    delay_topic_summary_path,
    delay_topic_examples_path,
    delay_unique_topics_path,
    delay_labeled_path,
    delay_var_by_year_path,
    delay_var_by_team_path,
    delay_var_by_referee_path,
]

print("Arquivos salvos:")
for output_path in output_paths:
    print(f"  {output_path}")
